In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
df = pd.read_csv("online_retail.csv")
print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
print(df.info()) 
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB
None
           Quantity         Price    Customer ID
count  1.067371e+06  1.067371e+06  824364.000000
mean   9.938898e+00  4.649388e+00   15324.638504
std    1.727058e+02  1.235531e+02    1697.464450
min   -8.099500e+04 -5.359436e+04   12346.000000
25%    1.000000e+00  1.250000e+00   13975.000000
50%    3.000000e+00  2.100000e+00   15255.000000
75%    1.000000e+01  4.150000e+00   1

In [4]:
# REMOVE DUPLICATE ROWS
before = len(df)
df.drop_duplicates(inplace=True)
print(f"Dropped   : {before - len(df):,} duplicate rows")
print(f"Remaining : {len(df):,} rows")

Dropped   : 34,335 duplicate rows
Remaining : 1,033,036 rows


In [5]:
#  HANDLE MISSING VALUES
print("Missing before:")
print(df.isnull().sum())
 
df.dropna(subset=["Customer ID"], inplace=True)
print(f"\nDropped rows with missing Customer ID. Remaining: {len(df):,}")
 
df.dropna(subset=["Description"], inplace=True)
print(f"Dropped rows with missing Description.  Remaining: {len(df):,}")
 

Missing before:
Invoice             0
StockCode           0
Description      4275
Quantity            0
InvoiceDate         0
Price               0
Customer ID    235151
Country             0
dtype: int64

Dropped rows with missing Customer ID. Remaining: 797,885
Dropped rows with missing Description.  Remaining: 797,885


In [6]:
# FIX DATA TYPES
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
print("InvoiceDate converted to datetime.")
 
df["Customer ID"] = df["Customer ID"].astype(str)
print("Customer ID converted to string (categorical key).")
 
print("\nDtypes after fix:")
print(df.dtypes)

InvoiceDate converted to datetime.
Customer ID converted to string (categorical key).

Dtypes after fix:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID            object
Country                object
dtype: object


In [7]:
# REMOVE CANCELLATIONS
before = len(df)
df = df[~df["Invoice"].str.startswith("C")]
print(f"Dropped   : {before - len(df):,} cancellation rows (Invoice starts with 'C')")
print(f"Remaining : {len(df):,} rows")

Dropped   : 18,390 cancellation rows (Invoice starts with 'C')
Remaining : 779,495 rows


In [8]:
# REMOVE NEGATIVE / ZERO QUANTITIES AND PRICES
before = len(df)
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]
print(f"Dropped   : {before - len(df):,} rows with Quantity ≤ 0 or Price ≤ 0")
print(f"Remaining : {len(df):,} rows")

Dropped   : 70 rows with Quantity ≤ 0 or Price ≤ 0
Remaining : 779,425 rows


In [9]:
# STRIP WHITESPACE FROM STRING COLUMNS
str_cols = ["Description", "Country", "StockCode"]
for col in str_cols:
    df[col] = df[col].str.strip()
print(f"Stripped whitespace from: {str_cols}")


Stripped whitespace from: ['Description', 'Country', 'StockCode']


In [10]:
# performing the churn window split 

snapshot_date = df['InvoiceDate'].max()           
churn_window_start = snapshot_date - pd.Timedelta(days=90)        
                                                                            
features_df = df[df['InvoiceDate'] < churn_window_start].copy()            
label_df = df[df['InvoiceDate'] >= churn_window_start].copy()                       
                                                                            
print("Snapshot date:", snapshot_date)                                       
print("Churn window start:", churn_window_start)                             
print("Features df:", features_df.shape)        
print("Label df:", label_df.shape)

Snapshot date: 2011-12-09 12:50:00
Churn window start: 2011-09-10 12:50:00
Features df: (620572, 8)
Label df: (158853, 8)


In [11]:
# building the churn labels

active_customers = set(label_df['Customer ID'].unique())
all_customers = features_df['Customer ID'].unique()

churn_labels = pd.DataFrame({
    'Customer ID': all_customers,
    'Churn': (~pd.Series(all_customers).isin(active_customers)).astype(int)
})

print(churn_labels['Churn'].value_counts())

Churn
1    2989
0    2292
Name: count, dtype: int64


In [12]:
# RFM features from features_df

features_df['Revenue'] = features_df['Quantity'] * features_df['Price']

rfm = features_df.groupby('Customer ID').agg(
    Recency = ('InvoiceDate', 'max'),
    Frequency = ('Invoice', 'nunique'),
    Monetary = ('Revenue', 'sum')
).reset_index()

rfm['Recency'] = (snapshot_date - rfm['Recency']).dt.days

print(rfm.shape)
print(rfm.head())

(5281, 4)
  Customer ID  Recency  Frequency  Monetary
0     12346.0      325         12  77556.46
1     12347.0      129          6   3402.39
2     12348.0      248          4   1709.40
3     12349.0      407          3   2671.14
4     12350.0      309          1    334.40


In [13]:
# additional features plus merge to the churn table

rfm['avg_basket_size'] = rfm['Monetary'] / rfm['Frequency']                                                                                                    
                                                                                                                                                                 
is_uk = features_df.groupby('Customer ID')['Country'].first().reset_index()                                                                                    
is_uk['is_uk'] = (is_uk['Country'] == 'United Kingdom').astype(int)                                                                                            
                                                                        
rfm = rfm.merge(is_uk[['Customer ID', 'is_uk']], on='Customer ID')                                                                                             
rfm = rfm.merge(churn_labels, on='Customer ID')

print(rfm.shape)
print(rfm.head())

(5281, 7)
  Customer ID  Recency  Frequency  Monetary  avg_basket_size  is_uk  Churn
0     12346.0      325         12  77556.46      6463.038333      1      1
1     12347.0      129          6   3402.39       567.065000      0      0
2     12348.0      248          4   1709.40       427.350000      0      0
3     12349.0      407          3   2671.14       890.380000      0      0
4     12350.0      309          1    334.40       334.400000      0      1


In [14]:
# null check and outliers

print(rfm.isnull().sum())                                                                                                                                      
print(rfm.describe())

Customer ID        0
Recency            0
Frequency          0
Monetary           0
avg_basket_size    0
is_uk              0
Churn              0
dtype: int64
           Recency    Frequency       Monetary  avg_basket_size        is_uk  \
count  5281.000000  5281.000000    5281.000000      5281.000000  5281.000000   
mean    297.282901     5.745692    2637.863286       369.292417     0.911380   
std     175.007174    11.464582   12225.544955       538.206215     0.284221   
min      90.000000     1.000000       2.900000         2.900000     0.000000   
25%     139.000000     1.000000     318.240000       174.668000     1.000000   
50%     254.000000     3.000000     783.580000       276.860000     1.000000   
75%     416.000000     6.000000    2078.540000       409.725000     1.000000   
max     738.000000   309.000000  456780.490000     14844.766667     1.000000   

             Churn  
count  5281.000000  
mean      0.565991  
std       0.495673  
min       0.000000  
25%       0.00

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler



In [16]:
X = rfm.drop(columns=['Customer ID', 'Churn'])
y = rfm['Churn']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y                                                                                                           
)

In [18]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
print("Train size:", X_train.shape)                                      
print("Test size:", X_test.shape)

Train size: (4224, 5)
Test size: (1057, 5)


In [20]:
from sklearn.ensemble import RandomForestClassifier      
from sklearn.metrics import classification_report, confusion_matrix                                                                                            
                                                                                                                                                                
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)                                                                      
rf.fit(X_train_scaled, y_train)                                                                                                                                
                                                        
y_pred = rf.predict(X_test_scaled)                                                                                                                             
                                            
print(confusion_matrix(y_test, y_pred))                                                                                                                        
print(classification_report(y_test, y_pred))

[[287 172]
 [142 456]]
              precision    recall  f1-score   support

           0       0.67      0.63      0.65       459
           1       0.73      0.76      0.74       598

    accuracy                           0.70      1057
   macro avg       0.70      0.69      0.70      1057
weighted avg       0.70      0.70      0.70      1057



Precision = TP / (TP + FP)                                                                                                                                     
  Recall    = TP / (TP + FN)                                               
  F1        = 2 × (Precision × Recall) / (Precision + Recall)                                                                                                    
  Accuracy  = (TP + TN) / (TP + TN + FP + FN)

In [22]:
precision = 456/(172+456)
recall = 456/(456+142)
f1 = 2 * (precision*recall)/(precision+recall)
accuracy = (456+287)/(287+172+142+456)

print("Precision: ", precision)
print("Recall: ", recall)
print("F1 score: ",f1)
print("Accuracy: ",accuracy)

Precision:  0.7261146496815286
Recall:  0.7625418060200669
F1 score:  0.7438825448613378
Accuracy:  0.7029328287606433
